**Load Data & Prep the Datas**

In [2]:
import pandas as pd
import datetime as dt

# 1. Load the dataset
df = pd.read_csv('clv_mock_data.csv')

# 2. Convert TransactionDate to datetime objects
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])

# 3. Check the basic info of the data
print("Data Overview:")
print(df.info())
df.head()

Data Overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1157 entries, 0 to 1156
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   CustomerID       1157 non-null   int64         
 1   TransactionDate  1157 non-null   datetime64[ns]
 2   Amount           1157 non-null   float64       
 3   Category         1157 non-null   object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 36.3+ KB
None


,CustomerID,TransactionDate,Amount,Category
0,1001,2026-01-29,1540.97,Grocery
1,1001,2025-09-08,1789.03,Grocery
2,1001,2025-01-07,1782.12,Home
3,1001,2025-12-01,1908.94,Fashion
4,1001,2025-06-06,1633.03,Electronics


**Calculate RFM Features**

In [3]:
# Create a reference date (usually one day after the last transaction in the data)
snapshot_date = df['TransactionDate'].max() + dt.timedelta(days=1)

# Group by CustomerID to calculate RFM
rfm = df.groupby('CustomerID').agg({
    'TransactionDate': lambda x: (snapshot_date - x.max()).days, # Recency
    'CustomerID': 'count',                                       # Frequency
    'Amount': 'mean'                                             # Monetary (Average spend)
})

# Rename columns for clarity
rfm.rename(columns={
    'TransactionDate': 'Recency',
    'CustomerID': 'Frequency',
    'Amount': 'Monetary'
}, inplace=True)

# Let's see the transformed customer-level data
print("RFM Table Created:")
rfm.head()

RFM Table Created:


,Recency,Frequency,Monetary
CustomerID,,,
1001,72,7,1653.938571
1002,167,9,1565.751111
1003,5,23,8933.526957
1004,28,16,9462.346875
1005,113,6,2486.668333


**Define Target and Split Data**

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

# 1. Calculate the Target Variable (Total Revenue per Customer)
target = df.groupby('CustomerID')['Amount'].sum()
rfm['CLV_Target'] = target

# 2. Define Features (X) and Target (y)
X = rfm[['Recency', 'Frequency', 'Monetary']]
y = rfm['CLV_Target']

# 3. Split the data (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")

Training rows: 160
Testing rows: 40


**Train the Linear Regression Model**

In [5]:
# 1. Initialize the model
model = LinearRegression()

# 2. Train the model using the training data
model.fit(X_train, y_train)

print("✅ Model Training Complete!")

✅ Model Training Complete!


**Evaluate the Model (Check Accuracy)**

In [6]:
# 1. Make predictions on the test set
y_pred = model.predict(X_test)

# 2. Calculate Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"R-squared Score (Accuracy): {round(r2 * 100, 2)}%")
print(f"Mean Absolute Error: Rs. {round(mae, 2)}")

# Show actual vs predicted for the first 5 customers in test set
comparison = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
comparison.head()

R-squared Score (Accuracy): 90.59%
Mean Absolute Error: Rs. 17565.31


,Actual,Predicted
CustomerID,,
1096,2072.20,-19145.169817
1016,24875.50,36896.306430
1031,3368.77,3513.802521
1159,17743.55,31767.487251
1129,1322.67,-14336.887831


**Create a Prediction Function**

In [7]:
def predict_clv(recency, frequency, monetary):
    # Prepare the input in a 2D array format as expected by Scikit-Learn
    new_data = pd.DataFrame({
        'Recency': [recency],
        'Frequency': [frequency],
        'Monetary': [monetary]
    })
    
    # Use the trained model to predict
    prediction = model.predict(new_data)
    
    return round(prediction[0], 2)

# --- TESTING THE FUNCTION ---
# Example: A customer who visited 10 days ago, bought 20 times, with avg spend of Rs. 8000
test_prediction = predict_clv(10, 20, 8000)

print(f"🚀 Predicted Lifetime Value for this Customer: Rs. {test_prediction}")

🚀 Predicted Lifetime Value for this Customer: Rs. 153179.2


**Saving the Model (Persistence)**

In [8]:
import joblib

# 1. Save the model to a file
joblib.dump(model, 'clv_linear_model.pkl')

# 2. Save the column names to keep them consistent during prediction
model_columns = list(X.columns)
joblib.dump(model_columns, 'model_columns.pkl')

print("✅ Model saved successfully as 'clv_linear_model.pkl'!")

✅ Model saved successfully as 'clv_linear_model.pkl'!
